# 🛡️ Face Anti-Spoofing — Week 10: 환경 설정
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---
## 체크리스트
- [ ] Cell 1: GPU 확인
- [ ] Cell 2: 라이브러리 설치
- [ ] Cell 3: 라이브러리 버전 확인
- [ ] Cell 4: GPU 동작 테스트
- [ ] Cell 5: Streamlit 동작 테스트
- [ ] Cell 6: 프로젝트 디렉토리 구조 생성

## ✅ Cell 1 — GPU 환경 확인
> **런타임 → 런타임 유형 변경 → T4 GPU** 로 설정 후 실행

In [ ]:
import subprocess

# GPU 할당 확인
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('✅ GPU 정상 할당됨')
else:
    print('❌ GPU가 할당되지 않았습니다.')
    print('런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요.')

Tue Apr 28 04:19:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Cell 2 — 필수 라이브러리 설치
> Colab에는 TensorFlow, NumPy, Matplotlib이 기본 설치되어 있어요.  
> OpenCV, Streamlit 등 추가 패키지만 설치합니다.

In [ ]:
# numpy 먼저 설치 후 런타임 재시작 필요
!pip install -q "numpy==1.26.4"
# 재시작 후 나머지 패키지 설치
!pip install -q streamlit mtcnn scikit-learn pyngrok

print('설치 완료!')

ERROR: Invalid requirement: 'numpy==1.26.4#': Expected end or semicolon (after version specifier)
    numpy==1.26.4#
         ~~~~~~~~^
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File 

## 🔍 Cell 3 — 라이브러리 버전 확인

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib
import sklearn
import streamlit

versions = {
    'TensorFlow' : tf.__version__,
    'OpenCV'     : cv2.__version__,
    'NumPy'      : np.__version__,
    'Matplotlib' : matplotlib.__version__,
    'Scikit-learn': sklearn.__version__,
    'Streamlit'  : streamlit.__version__,
}

print('=' * 40)
print('       라이브러리 버전 현황')
print('=' * 40)
for lib, ver in versions.items():
    status = '✅' if ver else '❌'
    print(f'  {status} {lib:<15} {ver}')
print('=' * 40)

# TF 버전 경고
tf_major = int(tf.__version__.split('.')[0])
if tf_major >= 2:
    print('\n✅ TensorFlow 2.x 확인됨 — 정상')
else:
    print('\n⚠️ TensorFlow 1.x 감지 — 2.x 이상 필요')

       라이브러리 버전 현황
  ✅ TensorFlow      2.19.0
  ✅ OpenCV          4.13.0
  ✅ NumPy           1.26.4
  ✅ Matplotlib      3.10.0
  ✅ Scikit-learn    1.6.1
  ✅ Streamlit       1.56.0

✅ TensorFlow 2.x 확인됨 — 정상


## ⚡ Cell 4 — GPU 동작 테스트 (TensorFlow)
> TensorFlow가 GPU를 실제로 사용하는지 확인합니다.

In [ ]:
import tensorflow as tf
import time

# GPU 인식 여부
gpus = tf.config.list_physical_devices('GPU')
print(f'감지된 GPU: {gpus}')

if gpus:
    print(f'\n✅ GPU {len(gpus)}개 감지됨: {gpus[0].name}')

    # 간단한 행렬 연산으로 GPU 동작 검증
    print('\n🔥 GPU 연산 테스트 중...')
    with tf.device('/GPU:0'):
        a = tf.random.normal([5000, 5000])
        b = tf.random.normal([5000, 5000])
        start = time.time()
        c = tf.matmul(a, b)
        elapsed = time.time() - start

    print(f'  행렬 곱셈 (5000×5000): {elapsed:.3f}초')
    print(f'  결과 shape: {c.shape}')
    print('\n✅ GPU 연산 정상 동작!')
else:
    print('\n⚠️ GPU 미감지 — CPU 모드로 실행 중')
    print('  런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재시작하세요.')

감지된 GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ GPU 1개 감지됨: /physical_device:GPU:0

🔥 GPU 연산 테스트 중...
  행렬 곱셈 (5000×5000): 0.098초
  결과 shape: (5000, 5000)

✅ GPU 연산 정상 동작!


## 🌐 Cell 5 — Streamlit 동작 테스트
> Colab에서 Streamlit을 실행하려면 `pyngrok`으로 외부 URL을 만들어야 합니다.  
> 아래 셀은 Hello World 앱으로 Streamlit이 정상 동작하는지 확인합니다.

In [ ]:
# Cell 5 - Streamlit 테스트 앱 생성
lines = [
    "import streamlit as st",
    "st.title('Face Anti-Spoofing System')",
    "st.subheader('Week 10 - 환경 설정 테스트')",
    "st.success('Streamlit 정상 동작 확인!')",
    "st.write('과목: AI Security & Application')",
    "st.write('학번: 32214391 조현수')",
]

with open('streamlit_hello.py', 'w') as f:
    f.write('\n'.join(lines))

print('streamlit_hello.py 생성 완료!')

streamlit_hello.py 생성 완료!


In [ ]:
# Streamlit 실행 (ngrok 터널로 외부 접속 URL 생성)
# ⚠️ ngrok 무료 계정 필요: https://ngrok.com → 가입 후 authtoken 복사

NGROK_TOKEN = '3CyGaGwA0OQ244SyRVYnWCoePYx_2kBqy37sYiB94iMugPWDi'  # 여기에 본인 ngrok authtoken 붙여넣기

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    import subprocess, time

    conf.get_default().auth_token = NGROK_TOKEN

    # Streamlit 백그라운드 실행
    proc = subprocess.Popen(
        ['streamlit', 'run', 'streamlit_hello.py',
         '--server.port', '8501',
         '--server.headless', 'true'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(3)

    tunnel = ngrok.connect(8501)
    print(f'\n🌐 Streamlit 접속 URL: {tunnel.public_url}')
    print('위 URL을 브라우저에서 열어보세요!')
else:
    print('⚠️ NGROK_TOKEN을 입력하지 않았습니다.')
    print('https://ngrok.com 에서 무료 가입 후 authtoken을 위 변수에 붙여넣으세요.')
    print('\n토큰 없이도 나머지 환경 설정은 모두 완료된 상태입니다. ✅')


🌐 Streamlit 접속 URL: https://strife-euphemism-confront.ngrok-free.dev
위 URL을 브라우저에서 열어보세요!


## 📁 Cell 6 — 프로젝트 디렉토리 구조 생성
> GitHub 저장소와 동일한 구조를 Colab 로컬에 미리 만들어둡니다.

In [ ]:
import os

import os

# 1. 프로젝트 루트 경로 설정
PROJECT_ROOT = '/content/face-anti-spoofing'

# 2. 생성할 디렉토리 목록
dirs = [
    'data/subset',
    'src',
    'app',
    'models',
    'reports',
    'logs',
    'notebooks',
]

# 3. 디렉토리 생성 및 .gitkeep 파일 추가
for d in dirs:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)
    # 빈 폴더도 Git에 추적되도록 .gitkeep 생성
    open(os.path.join(path, '.gitkeep'), 'w').close()

# 4. README.md 초기 파일 생성 (프로젝트 루트에)
readme_path = os.path.join(PROJECT_ROOT, 'README.md')
with open(readme_path, 'w') as f:
    f.write("# Face Anti-Spoofing Project\n\n")
    f.write("## 🛡️ 프로젝트 개요\n")
    f.write("- **과목:** AI Security & Application (단국대학교)\n")
    f.write("- **목표:** 멀티태스크 학습 및 XAI 기반의 위조 공격 방어 시스템 구현\n")
    f.write("- **현재 진행도:** 🌱 Week 10 (Environment Setup)\n\n")
    f.write("## 📁 폴더 구조\n")
    f.write("- `src/`: 소스 코드\n")
    f.write("- `data/subset/`: CelebA-Spoof 데이터셋 (6만 장)\n")
    f.write("- `app/`: Streamlit 데모 앱\n")
    f.write("- `reports/`: 분석 결과 및 그래프\n")

print('✅ 프로젝트 디렉토리 및 README.md 생성 완료!\n')

# 5. 생성된 구조 확인
for root, subdirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

✅ 프로젝트 디렉토리 및 README.md 생성 완료!

face-anti-spoofing/
  reports/
  notebooks/
  data/
    subset/
  app/
  logs/
  models/
  src/


## 🎉 Week 10-A 완료!

| 항목 | 상태 |
|------|------|
| T4 GPU 확인 | ✅ |
| TF / OpenCV / NumPy / Matplotlib / Streamlit 설치 | ✅ |
| GPU 연산 동작 테스트 | ✅ |
| Streamlit Hello World 실행 | ✅ |
| 프로젝트 디렉토리 구조 생성 | ✅ |

### 다음 단계
- **Week 10-B:** CelebA-Spoof 데이터셋 다운로드 스크립트
- **Week 10-C:** GitHub 저장소 초기화 + README.md 작성

# ✅ (04.29) 설치한 패키지 유지 여부 확인

In [ ]:
import numpy as np
import tensorflow as tf
print(f'numpy: {np.__version__}')
print(f'tensorflow: {tf.__version__}')
print('환경 OK!')

numpy: 2.0.2
tensorflow: 2.20.0
환경 OK!


# 📁 CelebA-Spoof 데이터 다운로드

그냥 CelebA-Spoof는 데이터가 너무 많아서 감당이 불가능
서브셋을 다운로드 하는 방향으로 정정

Kaggle에서 데이터셋을 다운로드 하기 위해서 사전에 API를 위한 kaggle.json을 사용

여기서 실제 데이터를 그냥 받아버리면 72GB 정도의 사진을 그냥 받을 수 있어서 AI에게 해결 방법을 질의함

# 🔑 해결책 제안
- 새로 찾은 데이터의 구조를 파악한다
- 이후 해당 데이터 중 필요한 데이터를 중심으로 다운로드를 수행한다

In [ ]:
import os

# 1. 새로 찾은 데이터셋의 구조(파일 목록)만 먼저 확인하기
print("🔍 데이터셋 파일 목록 확인 중...")
!kaggle datasets files mabdullahsajid/celeba-spoofing

# 2. 메타데이터(metas.zip 등)가 있다면 그것만 우선적으로 받는 것이 전략입니다.
# (전체 이미지 zip을 받기 전에 용량을 꼭 체크해야 합니다.)

🔍 데이터셋 파일 목록 확인 중...
Next Page Token = CfDJ8OqP5ZkTT9ZGj66XXRbxXyCCjoiLSI8b5X9X_vjKq11SMxSeMoKNLjbNII13aDgHDEBKKn6WJ0SdRyKmqsvlhcD_5QPI_VpH76du2qbf0vCfJlQLjTe7TfGon1gYdhbb2pQ80gdMAg4T2R1n_0WyvP1y6yeibA57qDnB1cRa6yX_ion6
name                                                size  creationDate                
-----------------------------------------------  -------  --------------------------  
CelebA_Spoof/Data/test/10001/live/496120.png     9429652  2024-06-08 11:30:16.235000  
CelebA_Spoof/Data/test/10001/live/496120_BB.txt       23  2024-06-08 11:30:16.192000  
CelebA_Spoof/Data/test/10001/live/497645.png      704776  2024-06-08 11:30:16.258000  
CelebA_Spoof/Data/test/10001/live/497645_BB.txt       22  2024-06-08 11:30:16.224000  
CelebA_Spoof/Data/test/10001/live/497900.png      738878  2024-06-08 11:30:16.247000  
CelebA_Spoof/Data/test/10001/live/497900_BB.txt       23  2024-06-08 11:30:16.242000  
CelebA_Spoof/Data/test/10001/live/498889.png      474111  2024-06-08 11:30:16.254000

index 40번(공격 유형)이 들어있는 라벨 파일을 탐색

- 초기에는 grep을 사용했으나 내부 구조를 알 수 없고, ai가 다양한 탐색을 사용하지만 제대로 탐색하지 못하는 것을 확인
- 따라서 직접 사이트에서 메타데이터의 위치를 찾음
```
/metas/intra_test/test_label.json

In [ ]:
import os

# 1. intra_test 폴더의 train_label.txt 다운로드
# (Kaggle 사이트 구조상 CelebA_Spoof/metas/ 가 앞에 붙습니다)
TARGET_FILE = 'CelebA_Spoof/metas/intra_test/train_label.txt'

print(f"📥 {TARGET_FILE} 다운로드 중...")
!kaggle datasets download -d mabdullahsajid/celeba-spoofing -f {TARGET_FILE}

# 2. 압축 풀기
if os.path.exists('train_label.txt.zip'):
    !unzip -o train_label.txt.zip
    print("✅ train_label.txt 확보 성공!")
else:
    # 혹시 경로가 CelebA_Spoof/Data/metas/... 일수도 있으니
    # 에러가 나면 사이트의 'Copy File Path'를 다시 확인해보세요!
    print("❌ 다운로드 실패. 경로를 다시 한번 확인해 주세요.")

📥 CelebA_Spoof/metas/intra_test/train_label.txt 다운로드 중...
Dataset URL: https://www.kaggle.com/datasets/mabdullahsajid/celeba-spoofing
License(s): CC0-1.0
100% 2.74M/2.74M [00:00<00:00, 227MB/s]

Archive:  train_label.txt.zip
  inflating: train_label.txt         
✅ train_label.txt 확보 성공!


**구글 드라이브 직접 연결하기!**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 오류 처리용 코드

In [ ]:
# 현재 어디에 어떤 폴더들이 있는지 확인합니다.
!ls -R /content/temp_data | grep ":$" | head -n 10

ls: cannot access '/content/temp_data': No such file or directory
